# Deploy Claims Gateway (GW1) & Interceptors

Deploy the AgentCore **Claims Gateway (GW1)** with its REQUEST and RESPONSE interceptor Lambdas.

GW1 is the claims path and is deployed on **both** IdP paths (Cognito and Okta) — the topology is identical. The authorizer (Cognito user-pool `allowedClients` vs Okta discovery `allowedAudience`) and the interceptor attachment are flag-branched **inside `create_gateway.py`** per `IDP_PROVIDER`, so the cells below stay IdP-agnostic.

## Prerequisites

- ✅ Run `04-deploy-mcp-server.ipynb` first
- ✅ MCP Server Runtime ARN saved to SSM

## What This Notebook Does

1. Deploys the claims **REQUEST** interceptor Lambda — validates the JWT and extracts caller identity (both IdP paths)
2. Deploys the claims **RESPONSE** interceptor Lambda — row/field filtering (per-persona `tools/list` + result scoping)
3. Creates the AgentCore Claims Gateway (GW1) and attaches both interceptors
4. Saves the Gateway ARN to SSM

> Note: there is no DynamoDB tenant-to-role table — access control is enforced by the two interceptors, not a tenant map.

## Next Notebook

- **06-deploy-agent.ipynb**

In [ ]:
# AWS Initialization - Load credentials and create session
from utils.notebook_init import init_aws

# This will:
# 1. Load credentials from .env file (if it exists)
# 2. Create and validate AWS session (env vars take precedence over SSO)
# 3. Return session, region, and account_id for use in this notebook
session, AWS_REGION, AWS_ACCOUNT_ID = init_aws()

# Initialize AWS clients
lambda_client = session.client("lambda", region_name=AWS_REGION)
ssm_client = session.client("ssm", region_name=AWS_REGION)

print("✅ Ready to proceed with AWS operations")
print(f"   Account ID: {AWS_ACCOUNT_ID}")
print(f"   Region: {AWS_REGION}")

## Step 1: Deploy Request Interceptor Lambda

In [ ]:
import subprocess

# Run deploy_interceptor.py to deploy Lambda function
result = subprocess.run(
    ["bash", "deploy.sh"],
    cwd="deployment/5a-gateway-setup/interceptor-request",
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("❌ Error:", result.stderr)
else:
    print("\n✅ Interceptor Lambda deployed!")

## Step 2: Deploy Response Interceptor Lambda

In [ ]:
# Deploy response interceptor Lambda
result = subprocess.run(
    ["bash", "deploy.sh"],
    cwd="deployment/5a-gateway-setup/interceptor-response",
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("❌ Error:", result.stderr)
else:
    print("\n✅ Response Interceptor Lambda deployed!")

## Step 3: Get Required ARNs

In [ ]:
# IdP-agnostic prerequisites (both paths). These two ARNs are what GW1 wires up.

# REQUEST interceptor Lambda ARN from SSM (saved by interceptor-request/deploy.sh)
INTERCEPTOR_ARN = ssm_client.get_parameter(Name="/app/lakehouse-agent/interceptor-lambda-arn")["Parameter"]["Value"]
print(f"✅ Interceptor ARN: {INTERCEPTOR_ARN}")

# MCP Server Runtime ARN from SSM (the claims MCP server GW1 fronts)
MCP_SERVER_RUNTIME_ARN = ssm_client.get_parameter(Name="/app/lakehouse-agent/mcp-server-runtime-arn")["Parameter"][
    "Value"
]
print(f"✅ MCP Server ARN: {MCP_SERVER_RUNTIME_ARN}")

# Note: the authorizer config (Cognito user-pool / allowedClients vs Okta
# discovery / allowedAudience) is loaded and set inside create_gateway.py per
# IDP_PROVIDER — no IdP-specific inputs are needed here.

## Step 4: Create AgentCore Gateway

This creates GW1 and configures it with the MCP server plus the REQUEST + RESPONSE interceptors. `create_gateway.py` reads `IDP_PROVIDER` once and sets the authorizer accordingly:

- **## [COGNITO]** — `customJWTAuthorizer` with `allowedClients` (Cognito access tokens carry no `aud`, so validation is by client ID).
- **## [OKTA]** — `customJWTAuthorizer` with `allowedAudience` against the Okta custom-auth-server discovery URL.

The subprocess call below is the same on both paths.

In [ ]:
# Create AgentCore Gateway
result = subprocess.run(
    [
        "python",
        "create_gateway.py",
        "--yes",  # Auto-confirm for notebook execution
    ],
    cwd="deployment/5a-gateway-setup",
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("❌ Error:", result.stderr)
else:
    print("\n✅ Gateway created!")
    print("\n📋 Gateway ARN saved to SSM Parameter Store")

## Step 5: Verify Gateway Configuration

The create_gateway.py script automatically saves the Gateway ARN to SSM.
Run this cell to verify the deployment.

In [ ]:
# Verify Gateway configuration in SSM
print("Verifying Gateway configuration in SSM...\n")

parameters_to_check = [
    "/app/lakehouse-agent/gateway-arn",
    "/app/lakehouse-agent/gateway-id",
    "/app/lakehouse-agent/gateway-url",
]

all_found = True
for param_name in parameters_to_check:
    try:
        response = ssm_client.get_parameter(Name=param_name)
        value = response["Parameter"]["Value"]
        print(f"✅ {param_name}")
        print(f"   Value: {value}")
    except ssm_client.exceptions.ParameterNotFound:
        print(f"❌ {param_name} - NOT FOUND")
        all_found = False
    except Exception as e:
        print(f"⚠️  {param_name} - ERROR: {e}")
        all_found = False

if all_found:
    print("\n✅ Gateway configuration verified in SSM!")
else:
    print("\n⚠️  Gateway parameters missing.")
    print("    The create_gateway.py script should have saved these automatically.")
    print("    Check the deployment output for errors.")

## Summary

✅ **Gateway & Interceptor Deployment Complete!**

**What was created:**
- REQUEST Interceptor Lambda (JWT validation, caller-identity extraction)
- RESPONSE Interceptor Lambda (row/field filtering by user persona)
- AgentCore Claims Gateway (GW1, routing)

All configuration saved to SSM Parameter Store.

**Next Steps:**
Run **06-deploy-agent.ipynb**